# Lab | Error Handling

Objective: Practice how to identify, handle and recover from potential errors in Python code using try-except blocks.

## Challenge 

Paste here your lab *functions* solutions. Apply error handling techniques to each function using try-except blocks. 

The try-except block in Python is designed to handle exceptions and provide a fallback mechanism when code encounters errors. By enclosing the code that could potentially throw errors in a try block, followed by specific or general exception handling in the except block, we can gracefully recover from errors and continue program execution.

However, there may be cases where an input may not produce an immediate error, but still needs to be addressed. In such situations, it can be useful to explicitly raise an error using the "raise" keyword, either to draw attention to the issue or handle it elsewhere in the program.

Modify the code to handle possible errors in Python, it is recommended to use `try-except-else-finally` blocks, incorporate the `raise` keyword where necessary, and print meaningful error messages to alert users of any issues that may occur during program execution.



In [ ]:
import re
import string
from typing import Iterable, List, Dict, Tuple, Callable

# ==========================
# Utilidades y validaciones
# ==========================

class InputTypeError(TypeError):
    pass

class MathDomainError(ValueError):
    pass

def assert_iterable_not_string(value, name="value"):
    if isinstance(value, (str, bytes)):
        raise InputTypeError(f"'{name}' no puede ser str/bytes; se esperaba una colección de elementos.")
    try:
        iter(value)
    except TypeError:
        raise InputTypeError(f"'{name}' debe ser iterable (lista, tupla, set, etc.).")

# =========================================
# 1) get_unique_list: únicos preservando orden
# =========================================
def get_unique_list(items: Iterable) -> List:
    """
    Devuelve los elementos únicos preservando el orden de aparición.
    Maneja entradas no iterables / strings con raise y mensajes claros.
    """
    try:
        assert_iterable_not_string(items, "items")
        vistos = set()
        resultado = []
        for x in items:
            if x not in vistos:
                resultado.append(x)
                vistos.add(x)
    except InputTypeError as e:
        print(f"Error: {e}")
        raise
    else:
        return resultado
    finally:
        # Punto de trazabilidad (opcional)
        pass

# =========================================
# 2) count_case: conteo de mayúsculas/minúsculas
# =========================================
def count_case(texto: str) -> Dict[str, int]:
    """
    Cuenta letras mayúsculas y minúsculas. Ignora no letras.
    """
    if not isinstance(texto, str):
        raise InputTypeError("count_case espera un str como entrada.")
    try:
        upper = sum(1 for c in texto if c.isalpha() and c.isupper())
        lower = sum(1 for c in texto if c.isalpha() and c.islower())
    except Exception as e:
        print(f"Error inesperado en count_case: {e}")
        raise
    else:
        return {"upper": upper, "lower": lower}
    finally:
        pass

# =========================================
# 3) remove_punctuation: elimina puntuación
# =========================================
_PUNCT_TABLE = str.maketrans("", "", string.punctuation)

def remove_punctuation(texto: str, keep: str = "") -> str:
    """
    Elimina puntuación estándar. Si 'keep' incluye caracteres, estos se conservan.
    """
    if not isinstance(texto, str):
        raise InputTypeError("remove_punctuation espera un str.")
    try:
        if keep:
            # Construir una clase de regex que quite toda puntuación excepto los 'keep'
            keep_escaped = re.escape(keep)
            pattern = rf"[{re.escape(string.punctuation).replace(keep_escaped, '')}]"
            return re.sub(pattern, "", texto)
        # vía tabla rápida
        return texto.translate(_PUNCT_TABLE)
    except Exception as e:
        print(f"Error en remove_punctuation: {e}")
        raise
    finally:
        pass

# =========================================
# 4) word_count: cuenta palabras (normalizado)
# =========================================
def word_count(texto: str, lowercase: bool = True, remove_punct: bool = True) -> Dict[str, int]:
    """
    Cuenta frecuencia de palabras. Normaliza espacios; opcionalmente quita puntuación y pasa a minúsculas.
    """
    if not isinstance(texto, str):
        raise InputTypeError("word_count espera un str.")
    try:
        if remove_punct:
            texto = remove_punctuation(texto)
        if lowercase:
            texto = texto.lower()
        # separa por cualquier espacio
        palabras = re.findall(r"\S+", texto)
        freqs: Dict[str, int] = {}
        for w in palabras:
            freqs[w] = freqs.get(w, 0) + 1
    except Exception as e:
        print(f"Error en word_count: {e}")
        raise
    else:
        return freqs
    finally:
        pass

# =========================================
# 5) Calculadora con operaciones múltiples y recursividad
# =========================================

def factorial(n: int) -> int:
    """Recursivo: factorial con validación."""
    if n < 0:
        raise MathDomainError("factorial no definido para números negativos.")
    if n in (0, 1):
        return 1
    return n * factorial(n - 1)

def potencia(base: float, exp: int) -> float:
    """Recursivo (exponente entero)."""
    if exp == 0:
        return 1.0
    if exp < 0:
        # a^(−n) = 1/a^n
        return 1.0 / potencia(base, -exp)
    # exp > 0
    return base * potencia(base, exp - 1)

def mcd(a: int, b: int) -> int:
    """Recursivo: máximo común divisor (Euclides)."""
    a, b = abs(a), abs(b)
    if b == 0:
        return a
    return mcd(b, a % b)

Operation = Callable[..., float]

def safe_div(a: float, b: float) -> float:
    if b == 0:
        raise MathDomainError("División por cero.")
    return a / b

OPERACIONES: Dict[str, Callable] = {
    "sum": lambda a, b: a + b,
    "sub": lambda a, b: a - b,
    "mul": lambda a, b: a * b,
    "div": safe_div,
    "pow": potencia,   # exp entero
    "fact": factorial, # unario
    "gcd": mcd,        # entero
}

def calcular(op: str, *args) -> float:
    """
    Enrutador de operaciones con manejo de errores y else/finally.
    Uso:
      calcular('sum', 2, 3) -> 5
      calcular('div', 10, 2) -> 5
      calcular('pow', 2, 8) -> 256
      calcular('fact', 5) -> 120
      calcular('gcd', 48, 18) -> 6
    """
    try:
        if op not in OPERACIONES:
            raise KeyError(f"Operación desconocida '{op}'. Disponibles: {', '.join(OPERACIONES)}")
        fn = OPERACIONES[op]

        # Validaciones según aridad conocida
        if fn is factorial:
            if len(args) != 1 or not isinstance(args[0], int):
                raise InputTypeError("factorial espera 1 entero.")
        elif fn is mcd:
            if len(args) != 2 or not all(isinstance(x, int) for x in args):
                raise InputTypeError("gcd espera 2 enteros.")
        elif fn is potencia:
            if len(args) != 2 or not isinstance(args[1], int):
                raise InputTypeError("pow espera (base: float|int, exp: int).")
        else:
            if len(args) != 2:
                raise InputTypeError(f"'{op}' espera 2 argumentos numéricos.")

        res = fn(*args)
    except (InputTypeError, MathDomainError, KeyError, TypeError, ValueError) as e:
        print(f"Error en calcular: {e}")
        raise
    else:
        return float(res) if isinstance(res, (int, float)) else res
    finally:
        # lugar para logs/telemetría/cierre de recursos si hiciera falta
        pass

# ==========================
# Mini tests / demostración
# ==========================
if __name__ == "__main__":
    # 1) get_unique_list
    print(get_unique_list([1, 2, 2, 3, 1, 4]))        # [1, 2, 3, 4]

    # 2) count_case
    print(count_case("Hola MUndo!!"))                  # {'upper': 3, 'lower': 7}

    # 3) remove_punctuation
    print(remove_punctuation("Hola, mundo! ¿Todo bien?"))  # "Hola mundo ¿Todo bien" (sin ! ,)

    # 4) word_count
    print(word_count("Hola hola, mundo Mundo!"))       # {'hola': 2, 'mundo': 2}

    # 5) Calculadora
    print(calcular("sum", 2, 5))                       # 7.0
    print(calcular("div", 10, 2))                      # 5.0
    print(calcular("pow", 2, 8))                       # 256.0
    print(calcular("fact", 5))                         # 120.0
    print(calcular("gcd", 48, 18))                     # 6.0


=== Inicialización del inventario ===

=== Pedidos del cliente ===
Error: Producto inválido. Elige entre: t-shirt, mug, hat, book, keychain.
Error: Producto inválido. Elige entre: t-shirt, mug, hat, book, keychain.
Error: Producto inválido. Elige entre: t-shirt, mug, hat, book, keychain.
Error: Producto inválido. Elige entre: t-shirt, mug, hat, book, keychain.
Error: Producto inválido. Elige entre: t-shirt, mug, hat, book, keychain.
Error: Producto inválido. Elige entre: t-shirt, mug, hat, book, keychain.
Error: Producto inválido. Elige entre: t-shirt, mug, hat, book, keychain.
Error: Producto inválido. Elige entre: t-shirt, mug, hat, book, keychain.
Error: Producto inválido. Elige entre: t-shirt, mug, hat, book, keychain.
Error: Producto inválido. Elige entre: t-shirt, mug, hat, book, keychain.
Error: Producto inválido. Elige entre: t-shirt, mug, hat, book, keychain.
Error: Producto inválido. Elige entre: t-shirt, mug, hat, book, keychain.
Error: Producto inválido. Elige entre: t-shir